In [0]:
spark.sql("USE CATALOG warehouse")
spark.sql("USE SCHEMA bronze")

In [0]:
df = spark.read.csv("/Volumes/warehouse/default/raw/", header=True, inferSchema=True)
display(df)
display(df.dtypes)

In [0]:
null_itemhash_df = df.filter(df["ItemHash"].isNull())
display(null_itemhash_df)

Have determined there are missing values in the `ItemHash`. For our intents and purposes, ItemHash is the primary key of the table, and unique identifier of the particular item. `SkuNumber` is the number that associates each item with identical items. With that being said, `ItemHash` can safely be imputed without worry.

In [0]:
import uuid
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType

existing_hashes = set([row.ItemHash for row in df.select("ItemHash").distinct().filter(col("ItemHash").isNotNull()).collect()])

def generate_unique_uuid(existing):
    while True:
        new_uuid = uuid.uuid4().hex[:12]
        if new_uuid not in existing:
            existing.add(new_uuid)
            return new_uuid

uuid_udf = udf(lambda x: generate_unique_uuid(existing_hashes) if x is None else x, StringType())

df_imputed = df.withColumn("ItemHash", uuid_udf(col("ItemHash")))
# display(df_imputed)
null_itemhash_df = df_imputed.filter(df_imputed["ItemHash"].isNull())
display(null_itemhash_df)

Currently with the given data, there are no missing values for the `SkuNumber`. However, for due diligence, one should impute the `SkuNumbers` should any missingness arise.

In [0]:
from pyspark.sql.functions import col, when

# Create a mapping DataFrame for (ItemName, Price, Category) -> SkuNumber
sku_map_df = df.filter(col("SkuNumber").isNotNull()) \
    .select("ItemName", "Price", "Category", "SkuNumber") \
    .dropDuplicates(["ItemName", "Price", "Category"]) \
    .withColumnRenamed("SkuNumber", "SkuNumber_map")

# Join original df with sku_map_df to impute missing SkuNumber
df_imputed = df.join(
    sku_map_df,
    on=["ItemName", "Price", "Category"],
    how="left"
).withColumn(
    "SkuNumber",
    when(col("SkuNumber").isNull(), col("SkuNumber_map")).otherwise(col("SkuNumber"))
).drop("SkuNumber_map")

The only values we should see in `SoldFlag` are 1s or 0s. If a value shows otherwise, it should be filled based on if any of the last 7 columns are not null. If they are not null, then it should be considered part of an order, and thus 1 for sold, else, 0.

In [0]:
from pyspark.sql.functions import col, when, expr

target_columns = ["OrderDate", "ShipDate", "OutDate", "Delivery_date", "OrderId", "CustomerId"]

# Cast each column to STRING to avoid implicit BIGINT casting on date/mixed types
array_expr = f"array({', '.join([f'cast(`{c}` as string)' for c in target_columns])})"
any_not_null_expr = expr(f"size(filter({array_expr}, x -> x is not null)) > 0")

df_imputed = df.withColumn(
    "SoldFlag",
    when(
        (~col("SoldFlag").isin(0, 1)) | col("SoldFlag").isNull(),
        when(any_not_null_expr, 1).otherwise(0)
    ).otherwise(col("SoldFlag"))
)

TODO: Ensure rows that actually have values of 0 or 1 are INDEED actual orders or not.

In [0]:
display(df.filter(col("ItemName").isNull()))

Even there are no null values in `ItemName`, it is prudent to write an imputation function. It should work similarly to the imputation function for `SkuNumber`, ensuring a check of another value that should ideally be reliable.

In [0]:
from pyspark.sql.functions import col, when

# Create a mapping DataFrame for (SkuNumber, Price, Category) -> ItemName
itemname_map_df = df.filter(col("ItemName").isNotNull()) \
    .select("SkuNumber", "Price", "Category", "ItemName") \
    .dropDuplicates(["SkuNumber", "Price", "Category"]) \
    .withColumnRenamed("ItemName", "ItemName_map")

# Join original df with itemname_map_df to impute missing ItemName
df_imputed_itemname = df.join(
    itemname_map_df,
    on=["SkuNumber", "Price", "Category"],
    how="left"
).withColumn(
    "ItemName",
    when(col("ItemName").isNull(), col("ItemName_map")).otherwise(col("ItemName"))
).drop("ItemName_map")

In [0]:
from pyspark.sql.functions import col, mean, stddev, abs, when

def flag_price_outliers(df):
    # Calculate mean and stddev for each Category
    stats_df = df.groupBy("Category").agg(
        mean("Price").alias("mean_price"),
        stddev("Price").alias("std_price")
    )

    # Join stats back to original df
    df_with_stats = df.join(stats_df, on="Category", how="left")

    # Flag items outside 3 stddev from mean
    df_flagged = df_with_stats.withColumn(
        "NeedsInspection",
        when(
            (col("Price").isNotNull()) &
            (col("std_price").isNotNull()) &
            (
                (col("Price") < col("mean_price") - 3 * col("std_price")) |
                (col("Price") > col("mean_price") + 3 * col("std_price")) |
                (col("Price") < 0 )
            ),
            1
        ).otherwise(0)
    )

    return df_flagged

df_validated = flag_price_outliers(df)
# display(df_validated)

In [0]:
display(df_validated.filter(col("NeedsInspection")==1))

The function works by flagging any items that have a price that falls outside of 3 standard deviations from the mean, or if the price is negative. If it is, then the item is flagged for inspection.

In [0]:
from pyspark.sql.functions import col, when

# Create a mapping DataFrame for SkuNumber -> Price
price_map_df = df.filter(col("Price").isNotNull()) \
    .select("SkuNumber", "Price") \
    .dropDuplicates(["SkuNumber"]) \
    .withColumnRenamed("Price", "Price_map")

# Join original df with price_map_df to impute missing Price
df_imputed_price = df.join(
    price_map_df,
    on=["SkuNumber"],
    how="left"
).withColumn(
    "Price",
    when(col("Price").isNull(), col("Price_map")).otherwise(col("Price"))
).drop("Price_map")

# display(df_imputed_price.filter(col("Price").isNull()))

This ensures that any null prices will be imputed with values of similar Sku'd items.

In [0]:
from pyspark.sql.functions import col, when, length, substring, expr

# Impute WarehouseId if null
df_imputed_warehouse = df.withColumn(
    "WarehouseId",
    when(
        col("WarehouseId").isNull() & col("SkuNumber").isNotNull(),
        expr("concat('WH', substring(SkuNumber, length(SkuNumber)-3, 4))")
    ).otherwise(col("WarehouseId"))
)

# Check WarehouseId validity
df_verified = df_imputed_warehouse.withColumn(
    "NeedsInspection",
    when(
        (
            (col("WarehouseId").isNull()) |
            (length(col("WarehouseId")) != 6) |
            (~col("WarehouseId").startswith("WH")) |
            (substring(col("WarehouseId"), 3, 4) != substring(col("SkuNumber"), length(col("SkuNumber"))-3, 4))
        ),
        1
    ).otherwise(0)
)

# display(df_verified)

In [0]:
from pyspark.sql.functions import col, regexp_extract, regexp_replace, when

def impute_category_from_itemname(df):
    # Extract category from ItemName (before first '-')
    itemname_category_col = regexp_extract(col("ItemName"), r"^([^-]+)", 1)
    
    # Normalize both for comparison: replace '&' with 'and' in Category
    normalized_category_col = regexp_replace(col("Category"), "&", "and")
    
    # Impute Category if mismatch or if Category is null
    df_imputed = df.withColumn(
        "Category",
        when(
            (col("ItemName").isNotNull()) &
            ((col("Category").isNull()) | (normalized_category_col != itemname_category_col)),
            itemname_category_col
        ).otherwise(col("Category"))
    )
    return df_imputed

df_verified = impute_category_from_itemname(df_verified)

display(df_verified)

In [0]:
df_verified.write.format("delta").mode("overwrite").saveAsTable("warehouse.bronze.bronze0")